# Wine Quality with Linear Regression on Amazon SageMaker

ANA680 Assignment 5, Problem 2
Jesus Adrian Alvarado Gadea

This notebook repeats Problem 1 with linear regression in two ways, following the module's Iris example:

- **(a) Without container technology:** scikit-learn's `LinearRegression` is trained and tested inside this notebook.
- **(b) With container technology:** SageMaker's built-in Linear Learner algorithm, which runs as a Docker container on AWS, is trained on data in S3 and deployed to a SageMaker endpoint for predictions.

The assignment asks for SageMaker Studio Lab, but Studio Lab no longer accepts new customers, and AWS names SageMaker Studio as its replacement ([AWS notice](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-lab-availability-change.html)). This notebook therefore runs in a SageMaker Studio JupyterLab space.

In [1]:
# The SageMaker Python SDK v2 has the Estimator API used in the Iris example (v3 replaced it)
%pip install -q "sagemaker>=2.250,<3"

Note: you may need to restart the kernel to use updated packages.


## Load the data

Like the Iris example, the data comes from the UCI repository. The notebook uses the CSV files in its folder if they are there, and otherwise downloads them.

In [2]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/"
for file_name in ["winequality-red.csv", "winequality-white.csv"]:
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(base_url + file_name, file_name)

red = pd.read_csv("winequality-red.csv", sep=";")
white = pd.read_csv("winequality-white.csv", sep=";")

# Same preparation as Problem 1: both files, with exact duplicates removed
wine = pd.concat([red, white], ignore_index=True).drop_duplicates().reset_index(drop=True)
FEATURES = wine.columns.drop("quality").tolist()
print("Wines:", len(wine))
wine.head()

Wines: 5318


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.66,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5


## Split the data

The split uses the same settings as Problem 1, so both problems test on the same 1,064 wines.

In [3]:
train_data, test_data = train_test_split(wine, test_size=0.2, random_state=42, stratify=wine["quality"])
print("Training wines:", len(train_data), "| Test wines:", len(test_data))


def evaluate(actual, predicted):
    """Error measures, plus how often the rounded prediction is exact or within one point."""
    score = np.clip(np.rint(predicted), 0, 10)
    return {"RMSE": mean_squared_error(actual, predicted) ** 0.5,
            "MAE": mean_absolute_error(actual, predicted),
            "R2": r2_score(actual, predicted),
            "Exact after rounding": np.mean(score == actual),
            "Within one point": np.mean(np.abs(score - actual) <= 1)}

Training wines: 4254 | Test wines: 1064


## Part (a): Without container technology

The model is trained and used directly in the notebook's own Python runtime.

In [4]:
linreg = LinearRegression()
linreg.fit(train_data[FEATURES], train_data["quality"])
pred_a = linreg.predict(test_data[FEATURES])

results = {"(a) scikit-learn LinearRegression": evaluate(test_data["quality"], pred_a)}
pd.DataFrame(results).round(3)

,(a) scikit-learn LinearRegression
RMSE,0.734
MAE,0.569
R2,0.304
Exact after rounding,0.529
Within one point,0.952


In [5]:
# Predictions next to the actual scores, as in the Iris example
combined_a = test_data[["quality"]].reset_index(drop=True).assign(prediction=pred_a.round(2))
combined_a.head(10)

,quality,prediction
0,5,5.60
1,6,6.37
2,6,6.14
3,7,6.19
4,5,5.51
5,4,5.33
6,6,5.21
7,4,5.11
8,6,6.44
9,6,5.18


In [6]:
# Actual scores against the rounded predictions
pd.crosstab(test_data["quality"].to_numpy(), np.clip(np.rint(pred_a), 0, 10).astype(int),
            rownames=["Actual quality"], colnames=["Predicted quality"])

Predicted quality,4,5,6,7
Actual quality,,,,
3,0,3,3,0
4,1,26,14,0
5,1,198,148,3
6,0,106,333,26
7,0,8,132,31
8,0,1,18,11
9,0,0,1,0


## Part (b): With container technology

SageMaker runs every training job and endpoint inside a Docker container. This part uses the built-in **Linear Learner** algorithm, whose container image is stored in Amazon ECR, and follows the same steps as the Iris example: save the data to S3, train the algorithm's container on it, deploy the trained model to an endpoint (another container), predict the test wines, and delete the endpoint.

The Iris example was written for the SageMaker SDK v1, so a few calls are updated for v2: `image_uris.retrieve` replaces `get_image_uri`, `instance_count` and `instance_type` replace `train_instance_count` and `train_instance_type`, `TrainingInput` replaces `s3_input`, and `CSVSerializer` and `JSONDeserializer` replace `csv_serializer` and `json_deserializer`.

In [7]:
from datetime import datetime

# SDK v2 is being replaced by v3, but it keeps the Estimator API from the Iris example
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

import sagemaker
from sagemaker.deserializers import JSONDeserializer
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer

session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()  # the SageMaker Studio domain's execution role
bucket = session.default_bucket()      # sagemaker-<region>-<account id>, created if needed
prefix = "wine-quality"
# This account has no training quota for CPU instance types, so training uses one ml.g5.xlarge GPU,
# which Linear Learner supports. The endpoint uses ml.m5.xlarge, an instance type in the free tier.
TRAIN_INSTANCE_TYPE = "ml.g5.xlarge"
ENDPOINT_INSTANCE_TYPE = "ml.m5.xlarge"

print("Region:", region, "| Execution role:", role.split("/")[-1])

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Region: us-east-2 | Execution role: AmazonSageMaker-ExecutionRole-20261017T125832


In [8]:
# Linear Learner reads CSV files with the label in the first column and no header row
columns = ["quality"] + FEATURES
train_data[columns].to_csv("wine-train.csv", index=False, header=False)
test_data[columns].to_csv("wine-test.csv", index=False, header=False)

train_s3 = session.upload_data("wine-train.csv", bucket=bucket, key_prefix=f"{prefix}/train")
test_s3 = session.upload_data("wine-test.csv", bucket=bucket, key_prefix=f"{prefix}/test")
print("Training and test data uploaded to S3")

Training and test data uploaded to S3


In [9]:
# The built-in algorithm's Docker image in Amazon ECR
container = sagemaker.image_uris.retrieve("linear-learner", region)
print("Linear Learner container image:", container)

linear = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type=TRAIN_INSTANCE_TYPE,
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
)

# Regression with squared loss, the same objective as ordinary linear regression
linear.set_hyperparameters(predictor_type="regressor", feature_dim=len(FEATURES),
                           loss="squared_loss", mini_batch_size=100, epochs=20)

# Train on the training data. The test channel makes the container also report test metrics in its log.
job_name = "wine-linear-learner-{}".format(datetime.now().strftime("%Y%m%d%H%M%S"))
linear.fit({"train": TrainingInput(train_s3, content_type="text/csv"),
            "test": TrainingInput(test_s3, content_type="text/csv")},
           job_name=job_name)

INFO:sagemaker:Creating training-job with name: wine-linear-learner-20261103050633


Linear Learner container image: 404615174143.dkr.ecr.us-east-2.amazonaws.com/linear-learner:1


2026-11-03 05:06:35 Starting - Starting the training job
2026-11-03 05:06:35 Pending - Training job waiting for capacity.

.

.


2026-11-03 05:06:48 Pending - Preparing the instances for training.

... (12 outputs left out)

#metrics {"StartTime": 1793682548.1334279, "EndTime": 1793682548.133458, "Dimensions": {"Algorithm": "Linear Learner", "Host": "algo-1", "Operation": "training", "Meta": "init_train_data_iter"}, "Metrics": {"Total Records Seen": {"sum": 4354.0, "count": 1, "min": 4354, "max": 4354}, "Total Batches Seen": {"sum": 44.0, "count": 1, "min": 44, "max": 44}, "Max Records Seen Between Resets": {"sum": 4254.0, "count": 1, "min": 4254, "max": 4254}, "Max Batches Seen Between Resets": {"sum": 43.0, "count": 1, "min": 43, "max": 43}, "Reset Count": {"sum": 2.0, "count": 1, "min": 2, "max": 2}, "Number of Records Since Last Reset": {"sum": 0.0, "count": 1, "min": 0, "max": 0}, "Number of Batches Since Last Reset": {"sum": 0.0, "count": 1, "min": 0, "max": 0}}}
#metrics {"StartTime": 1793682548.970033, "EndTime": 1793682548.970075, "Dimensions": {"Algorithm": "Linear Learner", "Host": "algo-1", "Operation": "training", "epoch": 0, "model": 0}, "Metrics": {"train_mse_objective": {"sum": 0.8521273431


2026-11-03 05:09:24 Uploading - Uploading generated training model


2026-11-03 05:09:42 Completed - Training job completed


Training seconds: 144
Billable seconds: 144


In [10]:
# Deploy the trained model to a SageMaker endpoint (this takes several minutes)
predictor = linear.deploy(initial_instance_count=1, instance_type=ENDPOINT_INSTANCE_TYPE,
                          serializer=CSVSerializer(), deserializer=JSONDeserializer())

INFO:sagemaker:Creating model with name: linear-learner-2026-11-03-05-10-21-331


INFO:sagemaker:Creating endpoint-config with name linear-learner-2026-11-03-05-10-21-331


INFO:sagemaker:Creating endpoint with name linear-learner-2026-11-03-05-10-21-331


-

-

-

... (2 outputs left out)

-

-

-

!

In [11]:
# Send the test wines (measurements only) to the endpoint and collect the predicted scores
response = predictor.predict(test_data[FEATURES].to_numpy())
pred_b = np.array([p["score"] for p in response["predictions"]])

results["(b) SageMaker Linear Learner container"] = evaluate(test_data["quality"], pred_b)
pd.DataFrame(results).round(3)

,(a) scikit-learn LinearRegression,(b) SageMaker Linear Learner container
RMSE,0.734,0.733
MAE,0.569,0.570
R2,0.304,0.306
Exact after rounding,0.529,0.523
Within one point,0.952,0.949


In [12]:
combined_b = test_data[["quality"]].reset_index(drop=True).assign(prediction=pred_b.round(2))
combined_b.head(10)

,quality,prediction
0,5,5.66
1,6,6.42
2,6,6.12
3,7,6.20
4,5,5.54
5,4,5.41
6,6,5.31
7,4,5.12
8,6,6.48
9,6,5.27


In [13]:
pd.crosstab(test_data["quality"].to_numpy(), np.clip(np.rint(pred_b), 0, 10).astype(int),
            rownames=["Actual quality"], colnames=["Predicted quality"])

Predicted quality,4,5,6,7
Actual quality,,,,
3,0,3,3,0
4,1,23,17,0
5,1,176,169,4
6,0,91,341,33
7,0,7,126,38
8,0,1,18,11
9,0,0,1,0


In [14]:
# Delete the model, then the endpoint, so they stop running up charges
# (the model has to go first, because the SDK looks it up through the endpoint)
predictor.delete_model()
predictor.delete_endpoint()
print("Model and endpoint deleted")

INFO:sagemaker:Deleting model with name: linear-learner-2026-11-03-05-10-21-331


INFO:sagemaker:Deleting endpoint configuration with name: linear-learner-2026-11-03-05-10-21-331


INFO:sagemaker:Deleting endpoint with name: linear-learner-2026-11-03-05-10-21-331


Model and endpoint deleted


## Comparison

Both parts fit a linear model to the same training wines and are scored on the same test wines. Part (a) solves the least-squares problem exactly in the notebook. Part (b) trains with stochastic gradient descent inside the Linear Learner container and serves predictions from a separate endpoint container, which is how a model is deployed in production. The results are close but not identical. Part (a) has a test RMSE of 0.734 and part (b) has 0.733, the same value the training container reported in its log for the test channel.

Both linear models do worse than the Random Forest from Problem 1 (test RMSE 0.68). That suggests the link between the lab measurements and quality is not purely linear.